# 📊 Multicollinearity Analysis Summary

This worksheet demonstrates the **'Mathematical Explosion'** that occurs in Linear Regression when two or more predictor variables are highly correlated.

### 🔹 1. The Setup: High Correlation
* **The Variables:** We created two features, `x1` and `x2`.
* **The Trap:** `x2` is almost a perfect duplicate of `x1` (multiplied by 2 with minimal noise).
* **The Result:** The correlation matrix shows a correlation of nearly **1.00**, indicating extreme multicollinearity.

### 🔹 2. The Determinant: Near-Singularity
* In Linear Regression, we must invert the matrix $(X^T X)$.
* **The Problem:** When variables are redundant, the **determinant** of this matrix drops dangerously close to zero ($10^{-10}$ in this case).
* **The Impact:** A near-zero determinant makes the matrix "ill-conditioned," meaning small changes in data lead to massive swings in results.

### 🔹 3. The Mathematical Explosion
* **Unstable Slopes:** Even though the true relationship only depended on `x1`, the model assigned massive, offsetting coefficients to both `x1` and `x2` (e.g., $+190,849$ and $-95,422$).
* **Inflated Standard Errors:** Because the model cannot distinguish the individual contribution of each variable, the **Standard Errors** became enormous ($>3,000,000$).
* **Statistical Significance:** Due to the high standard errors, the model might incorrectly suggest that *none* of the features are statistically significant, despite a high overall $R^2$.

### 🔹 4. Performance Metrics
* **RSS vs. TSS:** Despite the coefficient instability, the model still "fits" the data points well in terms of prediction.
* **R-squared ($R^2$):** We achieved a high $R^2$ (approx. $0.94$), proving that a high $R^2$ does **not** mean your individual feature coefficients are reliable or meaningful.

In [7]:
import numpy as np

# Set print options so we can see the full massive numbers without scientific notation
np.set_printoptions(suppress=True, precision=6)

# 1. Generate the Data
np.random.seed(42)

# We use standard, smaller numbers so the raw values don't inflate the determinant
x1 = np.random.normal(loc=0, scale=1, size=100)

# The Trap: x2 is exactly 2 * x1, with microscopic noise added (1e-8)
x2 = (x1 * 2) + np.random.normal(loc=0, scale=0.00000001, size=100)

# Calculate the correlation matrix
features = np.column_stack((x1, x2))
corr_matrix = np.corrcoef(features, rowvar=False)

print("--- The Correlation Matrix ---")
print(corr_matrix,'\n')

# True relationship: y = 10 + 5*x1 (x2 is completely ignored in reality)
y = 10 + (5 * x1) + np.random.normal(loc=0, scale=1, size=100)

# 2. Build the Matrix
ones = np.ones(100)
X = np.column_stack((ones, x1, x2))

# 3. Calculate the Determinant
X_T_X = X.T @ X
determinant = np.linalg.det(X_T_X)

print("--- 1. The Determinant ---")
print(f"Determinant of (X^T * X): {determinant:.10f}")
print("Notice how dangerously close to absolute zero this is.\n")

# 4. Invert the Matrix and Calculate Slopes
X_T_X_inv = np.linalg.inv(X_T_X)
beta = X_T_X_inv @ X.T @ y

# 5. Calculate Standard Errors
y_pred = X @ beta
rss = np.sum((y - y_pred)**2)
residual_variance = rss / (100 - 3) # n - features - 1
se_beta = np.sqrt(residual_variance * np.diag(X_T_X_inv))

print("--- 2. The Mathematical Explosion ---")
feature_names = ["Intercept", "x1 (The real feature)", "x2 (The duplicate)"]
for name, b, se in zip(feature_names, beta, se_beta):
    print(f"{name:25} | Slope: {b:15.2f} | Standard Error: {se:15.2f},",'\n')

# 3. Calculate the R-squared components

# Calculate Residual Sum of Squares (RSS)
rss = np.sum((y - y_pred)**2)

# Calculate Total Sum of Squares (TSS)
y_mean = np.mean(y)
tss = np.sum((y - y_mean)**2)

# Calculate R-squared
r_squared = 1 - (rss / tss)

print(f"Residual Sum of Squares (RSS): {rss:.4f}")
print(f"Total Sum of Squares (TSS): {tss:.4f}")
print(f"R-squared (R²): {r_squared:.4f}")

--- The Correlation Matrix ---
[[1. 1.]
 [1. 1.]] 

--- 1. The Determinant ---
Determinant of (X^T * X): 0.0000000009
Notice how dangerously close to absolute zero this is.

--- 2. The Mathematical Explosion ---
Intercept                 | Slope:           10.08 | Standard Error:            0.12, 

x1 (The real feature)     | Slope:       190849.37 | Standard Error:      7190868.75, 

x2 (The duplicate)        | Slope:       -95422.37 | Standard Error:      3595434.38, 

Residual Sum of Squares (RSS): 142.5557
Total Sum of Squares (TSS): 2343.7398
R-squared (R²): 0.9392
